## Custom Multi-Agent Workflows with LangChain

![Custom-workflow](./Images/Custom-workflow.png)

### Installing Utilities and Libraries

In [ ]:
%pip install -U \
    databricks-langchain==0.20.0 \
    langgraph==1.2.11 \
    mlflow

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Setup MLflow Tracing

In [ ]:
import mlflow
import os
import os
from dotenv import load_dotenv
from typing import Literal
from langchain.agents import AgentState, create_agent
from langchain.messages import AIMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing_extensions import NotRequired

# Enable auto-tracing for OpenAI
mlflow.openai.autolog()

# Set up MLflow tracking to Databricks
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/Custom-Workflow-Tracing")

### Instantiate the ChatDatabricks Class

In [ ]:
import json
from databricks_langchain import ChatDatabricks

model = ChatDatabricks(
    endpoint="databricks-gpt-oss-120b",
    temperature=0.1,
    max_tokens=20000,
)

### Define the Workflow State

In [ ]:
from typing import TypedDict

class VacationState(TypedDict):

    query: str

    location: str

    destination: str

    weather: str

    cuisine: str

    itinerary: str

### Create the Location Picker Agent

In [ ]:
from mlflow.entities import SpanType

@mlflow.trace(name="location_picker_agent", span_type=SpanType.AGENT)
def location_picker(state):

    print("\n===================================")
    print("Executing Location Picker Agent")
    print("===================================\n")

    response = model.invoke(f"""
You are a travel consultant.

Based on the user's request, identify the most appropriate vacation region.

User Request:

{state["query"]}

Return only the recommended location.
""")

    print("Location Picker Agent Output:{}".format(response.content))
    print("===================================\n")

    return {
        "location": response.content
    }

### Create the Destination Recommender Agent

In [ ]:
@mlflow.trace(name="destination_agent", span_type=SpanType.AGENT)
def destination_agent(state):

    print("\n===================================")
    print("Executing Destination Agent")
    print("===================================\n")

    response = model.invoke(f"""
You are a travel expert.

Recommend the best tourist destinations in:

{state["location"]}
""")

    print("Destination Agent Output:{}".format(response.content))
    print("===================================\n")

    return {
        "destination": response.content
    }

### Create the Weather Agent

In [ ]:
@mlflow.trace(name="weather_agent", span_type=SpanType.AGENT)
def weather_agent(state):

    print("\n===================================")
    print("Executing Weather Agent")
    print("===================================\n")

    response = model.invoke(f"""
You are a weather expert.

Describe the typical weather for:

{state["location"]}

Include the best season to visit.
""")

    print("Weather Agent Output:{}".format(response.content))
    print("===================================\n")

    return {
        "weather": response.content
    }

### Create the Cuisine Agent

In [ ]:
@mlflow.trace(name="cuisine_agent", span_type=SpanType.AGENT)
def cuisine_agent(state):

    print("\n===================================")
    print("Executing Cusisine Agent")
    print("===================================\n")

    response = model.invoke(f"""
You are a culinary expert.

Recommend local cuisine for:

{state["location"]}
""")

    print("Cuisine Agent Output:{}".format(response.content))
    print("===================================\n")

    return {
        "cuisine": response.content
    }

### Create the Itinerary Planner Agent

In [ ]:
@mlflow.trace(name="itinerary_planner_agent", span_type=SpanType.AGENT)
def itinerary_agent(state):

    print("\n===================================")
    print("Executing Itinerary Planner Agent")
    print("===================================\n")

    prompt = f"""
Create a detailed 5-day vacation itinerary.

Location

{state["location"]}

Destinations

{state["destination"]}

Weather

{state["weather"]}

Cuisine

{state["cuisine"]}
"""

    response = model.invoke(prompt)

    print("Itinerary Planner Agent Output:{}".format(response.content))
    print("===================================\n")

    return {
        "itinerary": response.content
    }

### Build the Custom Workflow

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(VacationState)

builder.add_node("location", location_picker)

builder.add_node("destination", destination_agent)

builder.add_node("weather", weather_agent)

builder.add_node("cuisine", cuisine_agent)

builder.add_node("itinerary", itinerary_agent)

builder.add_edge(START, "location")

# Fan-out
builder.add_edge("location", "destination")
builder.add_edge("location", "weather")
builder.add_edge("location", "cuisine")

# Fan-in
builder.add_edge("destination", "itinerary")
builder.add_edge("weather", "itinerary")
builder.add_edge("cuisine", "itinerary")

builder.add_edge("itinerary", END)

graph = builder.compile()
     

### Generate the Mermaid Diagram for the Custom Workflow

In [ ]:
png = graph.get_graph().draw_mermaid_png()

with open("vacation_workflow.png", "wb") as f:
    f.write(png)

### Execute the Workflow

In [ ]:
@mlflow.trace
def execute_workflow(user_query: str):
    response = graph.invoke(
        {
            "query": user_query
        }
    )

    return response["itinerary"]

In [ ]:
user_query = """ Help me plan a vacation to India.

    I enjoy historical places,
    prefer warm weather,
    and love trying local food. """

print(execute_workflow(user_query))